# Dataset verification (S5.10)

Round-trip verification: an independent model family reconstructs intents from prompts; field-by-field comparison routes examples into ACCEPT / REVIEW queues.

In [ ]:
# Havyn Stage 5 - environment setup and verification gate.
# Every notebook runs this first: no notebook trains or generates data on an
# unverified environment.
import sys, shutil, os
if not shutil.which("git"):
    raise RuntimeError("git missing")
!git clone --depth 1 https://github.com/HamzaNinjaCoder1/MCP-Packet-Tracer.git /content/MCP-Packet-Tracer 2>/dev/null || git -C /content/MCP-Packet-Tracer pull --ff-only
import sys
sys.path.insert(0, "/content/MCP-Packet-Tracer/havyn-stage5/scripts")
os.chdir("/content/MCP-Packet-Tracer")
from colab_setup import setup, save_report
env = setup(run_tests=True, use_drive=True)
assert not env.errors, f"environment verification failed: {env.errors}"
assert env.tests_passed, "offline suite failed - STOP, do not continue"
from env_report import build_report, recommend_config
report = build_report(repo="/content/MCP-Packet-Tracer", dataset_version="v0")
import json, pathlib
out_dir = pathlib.Path(env.drive_dir or "/content/havyn-artifacts")
out_dir.mkdir(parents=True, exist_ok=True)
(out_dir / "env-report.json").write_text(json.dumps(report, indent=2))
print(json.dumps(recommend_config(report.get("gpu", {})), indent=2))
print(json.dumps(report, indent=2)[:2000])


In [ ]:
# Requires INDEPENDENT_MODEL_ID via Colab secrets:
#   from google.colab import userdata; os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
#   os.environ['INDEPENDENT_MODEL_ID'] = userdata.get('INDEPENDENT_MODEL_ID')
!PYTHONPATH=havyn-stage5/scripts:. python havyn-stage5/scripts/roundtrip_verify.py \
  --prompts havyn-stage5/datasets/havyn-prompts-v1/prompts.jsonl \
  --out havyn-stage5/datasets/havyn-roundtrip-v1